# 20 · Deep learning et premiers pas en PyTorch — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch20_deep_learning/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch20_deep_learning/03_notebook.ipynb`, créée par `python tools/start_chapter.py 20`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 20.Q1 à 20.Q12, 20.R1 à 20.R3, 20.1 à 20.4, 20.6, 20.7, 20.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 20.13 à 20.17 | tes premiers tenseurs PyTorch, prédire les formes des couches utilitaires, l'autograd sur le réseau minuscule du ch. 18, puis ta librairie regularization.py : dropout, layer norm, pénalité L2 et early stopping (*prochaine session de génération*) | | | |
| B | 20.18 à 20.22 | ta classe nn.Module, Dataset et DataLoader, le zoo des couches, le plan de VGG16, la batchnorm de ta librairie (*prochaine session de génération*) | | | |
| C | 20.23 à 20.27 | la boucle d'entraînement complète sur MNIST, une chasse aux bugs, des courbes d'entraînement et de validation, un modèle ImageNet face à l'inconnu, la sauvegarde (*prochaine session de génération*) | | | |
| D | 20.28 à 20.31 | dropout, batchnorm, AdamW et early stopping sur Fashion-MNIST, une ablation, un réseau de 30 couches, le défi des 90 % (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="20")  # your own library

## Objectifs et rappel express

- Prévoir la forme d'un tenseur couche après couche, et dimensionner la sortie d'un réseau.
- Calculer à la main un dropout inversé, une batchnorm (entraînement et évaluation) et une layer norm, puis les programmer dans `mylearn.nn.regularization` et les retrouver dans PyTorch.
- Écrire un `nn.Module`, un `DataLoader` et une boucle d'entraînement PyTorch, et mener une ablation de régularisation.

**Rappel express.** Dropout inversé : $\tilde a = a\,m/(1 - p)$, $m \sim \mathrm{Bernoulli}(1 - p)$, un nouveau masque à chaque passe avant, l'identité en évaluation. Batchnorm : $\hat x = (x - \mu_B)/\sqrt{\sigma_B^2 + \epsilon}$ (variance biaisée), $y = \gamma\hat x + \beta$ ; statistiques glissantes $\mu^{\text{run}} \leftarrow (1 - \rho)\,\mu^{\text{run}} + \rho\,\mu_B$ avec $\rho = 0{,}1$ et la variance non biaisée. Layer norm : la même formule, sur les features de chaque exemple. Pénalité L2 : $\frac{\lambda}{2}\sum\lVert\mathbf{W}\rVert^2$, de gradient $\lambda\mathbf{W}$. Boucle PyTorch : `model.train()`, puis pour chaque mini-batch `logits = model(X)`, `loss = loss_fn(logits, y)`, `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` ; `model.eval()` et `torch.no_grad()` pour mesurer.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 20.1 à 20.4, 20.6, 20.7 et 🧮 20.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une forme : entre guillemets (`"(3, 5)"`, ou `"(7,)"` pour une seule dimension). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 20.5 et 20.8, la réflexion (🗣️ 20.10, ⚖️ 20.11, 📄 20.12) et l'entretien se corrigent avec `05_solutions.md`.

**20.Q1 — Deep learning et machine learning : qui contient qui ?**

In [ ]:
# 20.Q1: the values you found on paper
answer_20_Q1a = ...  # a) True or False
answer_20_Q1b = ...  # b) True or False
answer_20_Q1c = ...  # c) the letter of your choice, e.g. "E"
answer_20_Q1d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_Q1a, answer_20_Q1b, answer_20_Q1c, answer_20_Q1d]):
    wb.check(f"20.Q1{letter}", answer)

**20.Q2 — Tenseurs : nombre de dimensions, taille, forme**

In [ ]:
# 20.Q2: the values you found on paper
answer_20_Q2a = ...  # a) a whole number
answer_20_Q2b = ...  # b) a whole number
answer_20_Q2c = ...  # c) a shape as a string, e.g. "(3, 5)"
answer_20_Q2d = ...  # d) True or False
answer_20_Q2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_20_Q2a, answer_20_Q2b, answer_20_Q2c, answer_20_Q2d, answer_20_Q2e]):
    wb.check(f"20.Q2{letter}", answer)

**20.Q3 — Couche d'entrée, couche de sortie : laquelle contient des neurones ?**

In [ ]:
# 20.Q3: the values you found on paper
answer_20_Q3a = ...  # a) True or False
answer_20_Q3b = ...  # b) True or False
answer_20_Q3c = ...  # c) a whole number
answer_20_Q3d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_Q3a, answer_20_Q3b, answer_20_Q3c, answer_20_Q3d]):
    wb.check(f"20.Q3{letter}", answer)

**20.Q4 — Couche dense et couche d'activation séparée**

In [ ]:
# 20.Q4: the values you found on paper
answer_20_Q4a = ...  # a) a whole number
answer_20_Q4b = ...  # b) a whole number
answer_20_Q4c = ...  # c) True or False
answer_20_Q4d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_Q4a, answer_20_Q4b, answer_20_Q4c, answer_20_Q4d]):
    wb.check(f"20.Q4{letter}", answer)

**20.Q5 — Vrai ou faux : le dropout tire ses neurones une fois par epoch**

In [ ]:
# 20.Q5: the values you found on paper
answer_20_Q5a = ...  # a) True or False
answer_20_Q5b = ...  # b) a whole number
answer_20_Q5c = ...  # c) a number
answer_20_Q5d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_Q5a, answer_20_Q5b, answer_20_Q5c, answer_20_Q5d]):
    wb.check(f"20.Q5{letter}", answer)

**20.Q6 — Vrai ou faux : la batchnorm n'a aucun paramètre**

In [ ]:
# 20.Q6: the values you found on paper
answer_20_Q6a = ...  # a) True or False
answer_20_Q6b = ...  # b) a whole number
answer_20_Q6c = ...  # c) a whole number
answer_20_Q6d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_Q6a, answer_20_Q6b, answer_20_Q6c, answer_20_Q6d]):
    wb.check(f"20.Q6{letter}", answer)

**20.Q7 — Convolution et pooling en un coup d'œil**

In [ ]:
# 20.Q7: the values you found on paper
answer_20_Q7a = ...  # a) a whole number
answer_20_Q7b = ...  # b) a whole number
answer_20_Q7c = ...  # c) a whole number
answer_20_Q7d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_20_Q7a, answer_20_Q7b, answer_20_Q7c, answer_20_Q7d]):
    wb.check(f"20.Q7{letter}", answer)

**20.Q8 — Récurrence n'est pas récursion**

In [ ]:
# 20.Q8: the values you found on paper
answer_20_Q8a = ...  # a) True or False
answer_20_Q8b = ...  # b) the letter of your choice, e.g. "E"
answer_20_Q8c = ...  # c) True or False

for letter, answer in zip("abc", [answer_20_Q8a, answer_20_Q8b, answer_20_Q8c]):
    wb.check(f"20.Q8{letter}", answer)

**20.Q9 — Aplatir, remodeler, rogner, rembourrer, suréchantillonner, bruiter**

In [ ]:
# 20.Q9: the values you found on paper
answer_20_Q9a = ...  # a) a whole number
answer_20_Q9b = ...  # b) a whole number
answer_20_Q9c = ...  # c) a whole number
answer_20_Q9d = ...  # d) the letter of your choice, e.g. "E"
answer_20_Q9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_20_Q9a, answer_20_Q9b, answer_20_Q9c, answer_20_Q9d, answer_20_Q9e]):
    wb.check(f"20.Q9{letter}", answer)

**20.Q10 — VGG16 : ce qu'il reconnaît et ce qu'il ignore**

In [ ]:
# 20.Q10: the values you found on paper
answer_20_Q10a = ...  # a) a whole number
answer_20_Q10b = ...  # b) a whole number
answer_20_Q10c = ...  # c) a whole number
answer_20_Q10d = ...  # d) the letter of your choice, e.g. "E"
answer_20_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_20_Q10a, answer_20_Q10b, answer_20_Q10c, answer_20_Q10d, answer_20_Q10e]):
    wb.check(f"20.Q10{letter}", answer)

**20.Q11 — Construire un réseau profond : par où commencer ?**

In [ ]:
# 20.Q11: the values you found on paper
answer_20_Q11a = ...  # a) the letter of your choice, e.g. "E"
answer_20_Q11b = ...  # b) the letter of your choice, e.g. "E"
answer_20_Q11c = ...  # c) True or False
answer_20_Q11d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_Q11a, answer_20_Q11b, answer_20_Q11c, answer_20_Q11d]):
    wb.check(f"20.Q11{letter}", answer)

**20.Q12 — Explication complète ou explication satisfaisante ?**

In [ ]:
# 20.Q12: the values you found on paper
answer_20_Q12a = ...  # a) True or False
answer_20_Q12b = ...  # b) the letter of your choice, e.g. "E"
answer_20_Q12c = ...  # c) the letter of your choice, e.g. "E"
answer_20_Q12d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_20_Q12a, answer_20_Q12b, answer_20_Q12c, answer_20_Q12d]):
    wb.check(f"20.Q12{letter}", answer)

**20.R1 — Que fait weight_decay dans AdamW ?**

In [ ]:
# 20.R1: the values you found on paper
answer_20_R1a = ...  # a) a number
answer_20_R1b = ...  # b) True or False
answer_20_R1c = ...  # c) a number

for letter, answer in zip("abc", [answer_20_R1a, answer_20_R1b, answer_20_R1c]):
    wb.check(f"20.R1{letter}", answer)

**20.R2 — Pourquoi calcule-t-on la loss à partir des logits ?**

In [ ]:
# 20.R2: the values you found on paper
answer_20_R2a = ...  # a) True or False
answer_20_R2b = ...  # b) a whole number
answer_20_R2c = ...  # c) a whole number
answer_20_R2d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_R2a, answer_20_R2b, answer_20_R2c, answer_20_R2d]):
    wb.check(f"20.R2{letter}", answer)

**20.R3 — Limiter l'overfitting d'un arbre, et l'équivalent pour un réseau**

In [ ]:
# 20.R3: the values you found on paper
answer_20_R3a = ...  # a) the letter of your choice, e.g. "E"
answer_20_R3b = ...  # b) the letter of your choice, e.g. "E"
answer_20_R3c = ...  # c) True or False

for letter, answer in zip("abc", [answer_20_R3a, answer_20_R3b, answer_20_R3c]):
    wb.check(f"20.R3{letter}", answer)

**Ex 20.1 — Suivre la forme d'un tenseur couche après couche**

In [ ]:
# Ex 20.1: the values you found on paper
answer_20_1a = ...  # a) a whole number
answer_20_1b = ...  # b) a shape as a string, e.g. "(3, 5)"
answer_20_1c = ...  # c) a shape as a string, e.g. "(3, 5)"
answer_20_1d = ...  # d) a shape as a string, e.g. "(3, 5)"
answer_20_1e = ...  # e) a whole number
answer_20_1f = ...  # f) a shape as a string, e.g. "(3, 5)"
answer_20_1g = ...  # g) a shape as a string, e.g. "(3, 5)"
answer_20_1h = ...  # h) a shape as a string, e.g. "(3, 5)"

for letter, answer in zip("abcdefgh", [answer_20_1a, answer_20_1b, answer_20_1c, answer_20_1d, answer_20_1e, answer_20_1f, answer_20_1g, answer_20_1h]):
    wb.check(f"20.1{letter}", answer)

**Ex 20.2 — Dimensionner la couche de sortie et choisir la loss**

In [ ]:
# Ex 20.2: the values you found on paper
answer_20_2a = ...  # a) a whole number
answer_20_2b = ...  # b) the letter of your choice, e.g. "E"
answer_20_2c = ...  # c) a whole number
answer_20_2d = ...  # d) the letter of your choice, e.g. "E"
answer_20_2e = ...  # e) the letter of your choice, e.g. "E"
answer_20_2f = ...  # f) a whole number
answer_20_2g = ...  # g) the letter of your choice, e.g. "E"
answer_20_2h = ...  # h) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdefgh", [answer_20_2a, answer_20_2b, answer_20_2c, answer_20_2d, answer_20_2e, answer_20_2f, answer_20_2g, answer_20_2h]):
    wb.check(f"20.2{letter}", answer)

**Ex 20.3 — Dropout inversé à la main**

In [ ]:
# Ex 20.3: the values you found on paper
answer_20_3a = ...  # a) a whole number
answer_20_3b = ...  # b) a whole number
answer_20_3c = ...  # c) a number, 4 decimals
answer_20_3d = ...  # d) a number, 4 decimals
answer_20_3e = ...  # e) a number, 4 decimals
answer_20_3f = ...  # f) a number
answer_20_3g = ...  # g) a number, 4 decimals
answer_20_3h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_20_3a, answer_20_3b, answer_20_3c, answer_20_3d, answer_20_3e, answer_20_3f, answer_20_3g, answer_20_3h]):
    wb.check(f"20.3{letter}", answer)

**Ex 20.4 — Batchnorm à la main sur un mini-batch de quatre exemples**

In [ ]:
# Ex 20.4: the values you found on paper
answer_20_4a = ...  # a) a number
answer_20_4b = ...  # b) a number
answer_20_4c = ...  # c) a number
answer_20_4d = ...  # d) a number
answer_20_4e = ...  # e) a number, 4 decimals
answer_20_4f = ...  # f) a number, 4 decimals
answer_20_4g = ...  # g) True or False
answer_20_4h = ...  # h) a number
answer_20_4i = ...  # i) a number
answer_20_4j = ...  # j) a number, 4 decimals
answer_20_4k = ...  # k) a number, 3 decimals

for letter, answer in zip("abcdefghijk", [answer_20_4a, answer_20_4b, answer_20_4c, answer_20_4d, answer_20_4e, answer_20_4f, answer_20_4g, answer_20_4h, answer_20_4i, answer_20_4j, answer_20_4k]):
    wb.check(f"20.4{letter}", answer)

**Ex 20.6 — Compter les 138 millions de paramètres de VGG16**

In [ ]:
# Ex 20.6: the values you found on paper
answer_20_6a = ...  # a) a whole number
answer_20_6b = ...  # b) a whole number
answer_20_6c = ...  # c) a whole number
answer_20_6d = ...  # d) a whole number
answer_20_6e = ...  # e) a whole number
answer_20_6f = ...  # f) a number, 1 decimal
answer_20_6g = ...  # g) a number of MB, 1 decimal
answer_20_6h = ...  # h) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdefgh", [answer_20_6a, answer_20_6b, answer_20_6c, answer_20_6d, answer_20_6e, answer_20_6f, answer_20_6g, answer_20_6h]):
    wb.check(f"20.6{letter}", answer)

**Ex 20.7 — Connexion résiduelle et LayerNorm à la main**

In [ ]:
# Ex 20.7: the values you found on paper
answer_20_7a = ...  # a) a number
answer_20_7b = ...  # b) a number
answer_20_7c = ...  # c) a number, 4 decimals
answer_20_7d = ...  # d) a number
answer_20_7e = ...  # e) True or False
answer_20_7f = ...  # f) a number of 1e-9, 4 decimals
answer_20_7g = ...  # g) a whole number
answer_20_7h = ...  # h) a number, 4 decimals

for letter, answer in zip("abcdefgh", [answer_20_7a, answer_20_7b, answer_20_7c, answer_20_7d, answer_20_7e, answer_20_7f, answer_20_7g, answer_20_7h]):
    wb.check(f"20.7{letter}", answer)

**Ex 20.9 — Fermi : entraîner VGG16 sur ImageNet chez soi ?**

In [ ]:
# Ex 20.9: the values you found on paper
answer_20_9a = ...  # a) a number of billions, 1 decimal
answer_20_9b = ...  # b) a number of 1e18, 2 decimals
answer_20_9c = ...  # c) a number of days, 1 decimal
answer_20_9d = ...  # d) a whole number of days
answer_20_9e = ...  # e) a number of GB, 2 decimals

for letter, answer in zip("abcde", [answer_20_9a, answer_20_9b, answer_20_9c, answer_20_9d, answer_20_9e]):
    wb.check(f"20.9{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu prévoir la forme d'un tenseur après une couche dense, un aplatissement ou un pooling, et choisir la sortie et la loss d'un réseau ?
2. Sais-tu calculer à la main un dropout inversé et une batchnorm, et dire ce que changent `model.train()` et `model.eval()` ?
3. Sais-tu écrire une boucle d'entraînement PyTorch complète et expliquer chacune de ses lignes ?

**Pour aller plus loin** : le tutoriel « Learn the Basics » de PyTorch et *Dive into Deep Learning*, cités dans la fiche. La suite : le ch. 21, les réseaux de convolution, en PyTorch.

*Les exercices de code 20.13 à 20.31 (parties A à D, ta librairie `regularization.py` et PyTorch) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 20` pour obtenir le notebook complet.*